![image_1779979724740.png](./image_1779979724740.png "image_1779979724740.png")

![image_1779979756242.png](./image_1779979756242.png "image_1779979756242.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date
from pyspark.sql import functions as f
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("PurchasesDataFrame") \
    .getOrCreate()
data = [
    (1, 1, 500, "2023-01-10"),
    (2, 1, 750, "2023-02-15"),
    (3, 2, 200, "2023-01-20"),
    (4, 2, 100, "2023-03-05"),
    (5, 3, 2000, "2023-01-05"),
    (6, 3, 1500, "2023-02-28"),
    (7, 4, 800, "2023-03-10"),
    (8, 4, 600, "2023-04-01"),
    (9, 5, 50, "2023-01-15"),
    (10, 5, 75, "2023-02-20"),
    (11, 6, 3000, "2023-01-01"),
    (12, 7, 400, "2023-03-15"),
    (13, 7, 450, "2023-04-10"),
    (14, 8, 1000, "2023-02-01")
]
columns = ["purchase_id", "customer_id", "amount", "purchase_date"]
df = spark.createDataFrame(data, columns)
df = df.withColumn("purchase_date", to_date("purchase_date", "yyyy-MM-dd"))
df.show()

In [0]:
window_spec = Window.orderBy(f.col("total_spending").desc())
df_result=(
    df
    .groupBy("customer_id")
    .agg(
        f.sum("amount").alias("total_spending")
    )
).withColumn("pct",f.percent_rank().over(window_spec))\
    .select(
        "customer_id",
        "total_spending",
        f.when(f.col("pct")<= 0.25,"Platinum").when(f.col("pct")<=0.50,"Gold").when(f.col("pct")<= 0.75,"Silver").otherwise("Bronze").alias("tier")
    )

display(df_result)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as f

window_spec = Window.orderBy(f.col("total_spending").desc())
df_result=(
    df
    .groupBy("customer_id")
    .agg(
        f.sum("amount").alias("total_spending")
    )
).withColumn("pct",f.percent_rank().over(window_spec))\
    .select(
        "customer_id",
        "total_spending",
        f.when(f.col("pct")<= 0.25,"Platinum").when(f.col("pct")<=0.50,"Gold").when(f.col("pct")<= 0.75,"Silver").otherwise("Bronze").alias("tier")
    )

display(df_result)